# Hairstyle Classification Benchmark

Compares LIG Vision against frontier vision-language models on a public labelled dataset.
Run `evaluate.py` first to populate `results/`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

results_dir = Path('../results')

results = {}
for p in sorted(results_dir.glob('*.json')):
    with open(p) as f:
        d = json.load(f)
    if 'n_images' not in d:   # skip internal/partial result files
        continue
    results[p.stem] = d

print('Loaded models:', list(results.keys()))

## Summary table

In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        'Model':    name,
        'Key F1':   r['metrics']['key']['macro_f1'],
        'Key mAP':  r['metrics']['key'].get('map'),
        'Base F1':  r['metrics']['base']['macro_f1'],
        'Images':   r['n_images'],
        'Errors':   r['n_errors'],
        'Date':     r['evaluated_at'],
    })

df = pd.DataFrame(rows).set_index('Model')
df

## Key head — per-class F1

In [ ]:
# collect all key classes across models
all_classes = set()
for r in results.values():
    all_classes.update(r['metrics']['key']['per_class'].keys())
all_classes = sorted(all_classes)

fig, ax = plt.subplots(figsize=(14, 6))
x    = np.arange(len(all_classes))
w    = 0.8 / len(results)

for i, (name, r) in enumerate(results.items()):
    vals = [r['metrics']['key']['per_class'].get(c, {}).get('f1', 0) for c in all_classes]
    ax.bar(x + i * w, vals, w, label=name)

ax.set_xticks(x + w * (len(results) - 1) / 2)
ax.set_xticklabels(all_classes, rotation=60, ha='right', fontsize=8)
ax.set_ylabel('F1')
ax.set_title('Key head — per-class F1')
ax.legend()
plt.tight_layout()
plt.show()

## Base head — per-class F1

In [ ]:
base_classes = sorted(set(
    c for r in results.values()
    for c in r['metrics']['base']['per_class']
))

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(base_classes))
w = 0.8 / len(results)

for i, (name, r) in enumerate(results.items()):
    vals = [r['metrics']['base']['per_class'].get(c, {}).get('f1', 0) for c in base_classes]
    ax.bar(x + i * w, vals, w, label=name)

ax.set_xticks(x + w * (len(results) - 1) / 2)
ax.set_xticklabels(base_classes, rotation=30, ha='right')
ax.set_ylabel('F1')
ax.set_title('Base head — per-class F1')
ax.legend()
plt.tight_layout()
plt.show()

## Average latency per model

In [ ]:
for name, r in results.items():
    latencies = [p['latency_ms'] for p in r['predictions'] if p.get('latency_ms')]
    if latencies:
        print(f"{name:<15}  avg {np.mean(latencies):.0f}ms  p95 {np.percentile(latencies, 95):.0f}ms")